In [52]:
import numpy as np

# КОНСТАНТЫ 
fs = 8000 # ЧАСТОТА ДИСКРЕТИЗАЦИИ
dt = 1/fs # ШАГ ДИСКРЕТИЗАЦИИ
N = 1000 # КОЛИЧЕСТВО ТОЧЕК 
t = np.arange(N)*dt #ВЕКТОР ВРЕМЕНИ

# ПАРАМЕТРЫ АВП
M = 4 # КОЛИЧЕСТВО ДАТЧИКОВ
R = 1 # РАДИУС

# ПАРАМЕТРЫ СИГНАЛА 
wavelength = 20*R # ДЛИНА ВОЛНЫ (ИЗ УСЛОВИЯ МНОГО МЕНЬШЕ)
f = 1000 # ЦЕНТРАЛЬНАЯ ЧАСТОТА СИГНАЛА 
A = 2 # АМПЛИТУДА СИГНАЛА 
phi_0 = np.pi/2 # ФАЗА СИГНАЛА (ТО ЧТО ОЦЕНИВАЕМ)

![АВП](visual1.png)

In [53]:
def get_gamma_array(M: int) -> np.ndarray:
    """Рассчет углов датчиков относительно оси x
    Функция принимает на вход число датчиков и возвращает 
    массив углов для каждого датчика"""
    
    return (2*np.pi/M)*np.arange(M)

def get_tau_array(M: int, 
                  R: float, 
                  phi_o: float, 
                  c: float = 343.26) -> np.ndarray:
    """Рассчёт задержек относительно каждого датчика"""
    
    return (R/c)*np.cos(phi_o - get_gamma_array(M))

In [54]:

def generate_signal(t: np.ndarray, 
                    f: float,
                    A: float = 1.0,) -> np.ndarray:
    """Считаем значение гармончиеского сигнала в каждый момент времени"""
    
    return  A*np.sin(t*(2*np.pi*f))
    

In [55]:
def generate_channel_matrix(
                      t: np.ndarray, 
                      f: float,
                      phi_0: float,
                      tau_array: np.ndarray,
                      M: int,
                      N: int,
                      A: float = 1.0) -> np.ndarray:
    """Функция возращает матрицу 3M x N """
    channel_matrix = np.zeros((3*M, N)) # Создаём нулевую матрицу размера M x N
    for idx, tau in enumerate(tau_array):
        p_t = generate_signal(t - tau, f, A)
        y0 = p_t # size (1 x N)
        y1 = p_t*np.cos(phi_0)
        y2 = p_t*np.sin(phi_0)
        sensor_matrix = np.vstack((y0, y1, y2)) 
        channel_matrix[idx*3:idx*3+3] = sensor_matrix
        
    return channel_matrix

In [ ]:
def generate_covariation_matrix(matrix: np.ndarray,
                                N: int) -> np.ndarray:
    return (matrix @ matrix.T) / N


In [ ]:
tau_array = get_tau_array(M, R, phi_0) # shape (M,)
channel_matrix = generate_channel_matrix(t, f, phi_0, tau_array, M,N) # (3M x N)
channel_matrix.shape

(12, 1000)

In [ ]:
W = generate_covariation_matrix(channel_matrix) # 3M x 3M
W.shape

(12, 12)